# Exploratory Data Analysis — Road Segmentation

This notebook explores the IDD (Indian Driving Dataset) for road segmentation.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Dataset Overview

In [ ]:
# Paths
IMAGE_DIR = Path('../data/raw/images/train')
MASK_DIR = Path('../data/raw/masks/train')

# Count files
images = list(IMAGE_DIR.glob('*'))
masks = list(MASK_DIR.glob('*'))

print(f'Total images: {len(images)}')
print(f'Total masks: {len(masks)}')
print(f'Image formats: {Counter([f.suffix for f in images])}')

## 2. Image Size Distribution

In [ ]:
sizes = []
for img_path in images[:100]:  # Sample first 100
    img = cv2.imread(str(img_path))
    if img is not None:
        sizes.append(img.shape[:2])

heights, widths = zip(*sizes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(heights, bins=30, edgecolor='black')
axes[0].set_title('Image Height Distribution')
axes[0].set_xlabel('Height (pixels)')

axes[1].hist(widths, bins=30, edgecolor='black')
axes[1].set_title('Image Width Distribution')
axes[1].set_xlabel('Width (pixels)')
plt.tight_layout()
plt.show()

## 3. Sample Images and Masks

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(3):
    img_path = images[i]
    mask_path = MASK_DIR / img_path.name
    
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else np.zeros(img.shape[:2])
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Image: {img_path.name}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title('Ground Truth Mask')
    axes[i, 1].axis('off')
    
    # Overlay
    overlay = img.copy()
    overlay[mask > 0] = (0.5 * overlay[mask > 0] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')
    
    # Mask statistics
    road_ratio = (mask > 0).sum() / mask.size * 100
    axes[i, 3].text(0.1, 0.5, f'Road pixels: {(mask > 0).sum():,}\nRoad ratio: {road_ratio:.1f}%',
                   fontsize=14, transform=axes[i, 3].transAxes)
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 4. Class Distribution

In [ ]:
road_ratios = []
for mask_path in masks[:100]:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is not None:
        road_ratios.append((mask > 0).sum() / mask.size * 100)

plt.figure(figsize=(10, 6))
plt.hist(road_ratios, bins=30, edgecolor='black')
plt.title('Road Pixel Ratio Distribution')
plt.xlabel('Road Pixels (%)')
plt.ylabel('Frequency')
plt.axvline(np.mean(road_ratios), color='red', linestyle='--', label=f'Mean: {np.mean(road_ratios):.1f}%')
plt.legend()
plt.show()